# Reproducing the IBB2011 comparison figures

Notebook driver for `ibb2011_comparison_figs.py` (see the **2026-09-11** entry
in `../research-notebook.md`): the same 11-figure set as
`rosolowsky2003_figs.py`, plus two direct-comparison figures (fig12, fig13),
run on the Rosolowsky+2003 BIMA CO catalogue (45 GMCs). The HI envelope
around each GMC is a fixed **140 pc x 140 pc sky-plane box** over a
**50 km/s** window centred on the GMC's catalogue position and VLSR (Imara,
Bigiel & Blitz 2011, Section 3), rather than the rest of the pipeline's
mass-scaled circular aperture.

This notebook calls straight into `ibb2011_comparison_figs.py`'s own
functions cell by cell (`analyse`, `fig1` ... `fig13`) instead of
reimplementing them, so the notebook and the script that already produced
`../paper/figures_ibb2011_comparisons/` stay one source of truth.

**Data.** The full-resolution cube
(`M33_A+B+C+D_HI_0p4kms_feathered_K_reimaged.fits`, ~111 GiB, 7874x6635x572)
is **not on this laptop**. Per the 2026-09-23 notebook entry it lives on
Hummingbird at `~/data/M33/`. To run the full-res cells: either point
`IBB_CUBE` below at a local mount of that path, or run this notebook's
kernel on Hummingbird itself (e.g. a remote Jupyter kernel over SSH) --
per the M33 data-transfer notes, don't route the 111 GiB file through this
laptop just to read it here.

The mid-res cube (`m33_C+D+tp_himidres_120pc.fits`, ~2 GB) used for the
fig12/fig13 second comparison series **is** present in `../data/`, so those
cells run locally as-is. Every cell that needs the full-res cube checks a
`HAVE_FULLRES` flag first and prints a message instead of failing if it's
absent.


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mekuria_imara_pipeline as mp
import rosolowsky2003_catalog as rc
import ibb2011_comparison_figs as icf

icf.FIGDIR.mkdir(parents=True, exist_ok=True) #icf.FIGDIR = "../paper/figures_ibb2011_comparisons"


def save(fig, stem):
    """Save a figure into ../paper/figures_ibb2011_comparisons/, matching
    the script's own main() naming, and show it inline."""
    path = icf.FIGDIR / f"{stem}.pdf"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("wrote", path)
    return fig


## 1. Catalogue and reference tables

Rosolowsky+2003 Table 2 (45 BIMA GMCs), the CO-side quantities derived from
it, Mekuria & Imara (2026) Table 2, Imara/Bigiel/Blitz (2011) Table 1, and
each cloud's catalogue VLSR (used to centre the fixed velocity window).


In [24]:
print(gmc)

    Omega_GMC  Phi_GMC  j_GMC_1e24  M_GMC_1e4   R_GMC_pc     R_kpc
1    0.081946    178.0    6.058508  27.189493  27.610714  0.180654
2    0.022393    -64.0         NaN   7.006523        NaN  0.303953
3    0.093681    -82.0         NaN   3.764699        NaN  0.510737
4    0.098766    -44.0    4.827873   4.392149  22.497619  0.542030
5    0.132992    -60.0   29.629888  14.640496  49.085714  0.566181
6    0.116368     19.0         NaN   3.032674        NaN  0.567588
7    0.081751    143.0    1.262189  10.352922  13.294048  0.583324
8    0.034128    -33.0         NaN   3.032674        NaN  0.633230
9    0.021513     -9.0    0.334480   5.542473  13.294048  0.691442
10   0.018580    -16.0    2.028969  12.548997  33.746429  0.765667
11   0.093192     24.0    9.466418  42.875738  32.723810  0.757052
12   0.103655     32.0         NaN   7.006523        NaN  0.734676
13   0.032172    -46.0    1.596669  11.503247  22.497619  0.822140
14   0.075297    -91.0    0.328169   3.869274   6.851548  0.81

In [2]:
cat = rc.load(path=str(icf.TAB2)) #icf.TAB2 = "../data/rosolowsky2003_tab2.csv"
gmc = icf.load_gmc_side(cat)
mi = icf.load_mi_tab2()
ibb = icf.load_ibb_tab1()
vlsr = icf._load_vlsr()
print(f"catalogue: {len(cat)} GMCs (Rosolowsky+2003)")


catalogue: 45 GMCs (Rosolowsky+2003)


## 2. Full-resolution cube -- HI envelopes via the box-cutout method

`FGND_CHAN` is overridden to 0: this cube's velocity axis runs only
-74 -> -310 km/s (entirely within the M33 disk), so the pipeline's default
60-channel Milky-Way-foreground trim would cut real disk emission instead of
foreground. See the 2026-09-11 notebook entry.

Edit `IBB_CUBE` below if the cube is reachable at a different local path.


In [3]:
icf.mp.FGND_CHAN = 0   # this cube carries no MW foreground to trim (see above)
HI_CUBE = icf.MIDRES_CUBE   #icf.FULLRES_CUBE

try:
    cube = mp.open_cube(path=str(HI_CUBE))
    HAVE_FULLRES = True
except (FileNotFoundError, OSError) as e:
    print(f"Full-res cube not found at {HI_CUBE}:\n  {e}")
    print("It lives on Hummingbird at ~/data/M33/ (111 GiB) -- see the ")
    print("2026-09-23 research-notebook entry. Point HI_CUBE at a local ")
    print("mount/copy, or run this notebook's kernel on a machine that has it.")
    cube = None
    HAVE_FULLRES = False
HAVE_FULLRES


True

In [4]:
if HAVE_FULLRES:
    tab, results = icf.analyse(cube, cat, vlsr)
    tab.to_csv(icf.OUT_CSV, index=False)
    print(f"catalog n={len(cat)}  kept n={len(tab)}  -> {icf.OUT_CSV}")
else:
    tab, results = None, None
    print("skipped -- needs the full-res cube")


catalog n=45  kept n=45  -> hi_table2_m33_rosolowsky2003_ibbcube.csv


In [5]:
# fig1's Sigma_HI background field: a single decimated pass over the cube
# (step=4 in velocity, chunk=8 channels/block -- this cube's planes are
# ~24x the mid-res cube's, so a smaller chunk keeps memory bounded).
if HAVE_FULLRES:
    sigma, _ = mp.field_moments(cube, step=4, chunk=8)
else:
    sigma = None


## 3. Mid-res cube -- the same box-cutout method, for the fig12/fig13 comparison

Run again on the original `m33_C+D+tp_himidres_120pc.fits` cube (present
locally), so fig12/fig13 can show whether the box-cutout method's agreement
with IBB 2011 depends on the cube's resolution rather than on the method
itself. This cube's velocity axis does reach the Milky Way foreground band,
so `FGND_CHAN` goes back to the pipeline's default (60).


In [6]:
icf.mp.FGND_CHAN = 60   # this cube's velocity axis does reach the MW foreground band
cube_mid = mp.open_cube(path=str(icf.MIDRES_CUBE))
tab_mid, results_mid = icf.analyse(cube_mid, cat, vlsr)
tab_mid.to_csv(icf.OUT_CSV_MIDRES, index=False)
print(f"catalog n={len(cat)}  kept n={len(tab_mid)}  -> {icf.OUT_CSV_MIDRES}")


catalog n=45  kept n=45  -> hi_table2_m33_rosolowsky2003_midrescube_boxcut.csv


## 4. Figures

### Figure 1 -- Σ_HI field, envelope boxes and HI/CO velocity gradients [A]

Grayscale Σ_HI background (this work's full-res cube), one 140 pc box per GMC, HI gradient arrows (pointing up-gradient) and CO gradient arrows (Rosolowsky+2003's tabulated position angle), cropped to the inner disk where the BIMA GMCs sit.

In [ ]:
# Circular 140 pc diameter mask for each velocity-channel plane.
# Insert immediately after the 140 pc box subcube is created.
ny, nx = subcube.shape[-2:]
y, x = np.ogrid[:ny, :nx]

cx = (nx - 1) / 2
cy = (ny - 1) / 2
radius_pix = 70.0 / cube.pixelscale_pc

circular_mask = (x - cx) ** 2 + (y - cy) ** 2 <= radius_pix ** 2

# Broadcast over all velocity channels.
subcube = np.where(circular_mask[None, :, :], subcube, np.nan)

In [7]:
if HAVE_FULLRES:
    fig1 = save(icf.fig1(cube, results, sigma, gmc), "fig1_field_gradients")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig1_field_gradients.pdf


### Figure 2 -- example aperture spectra [B]

2x3 grid of stacked HI spectra over each envelope: three most planar clouds (top row, high ρ) and three least planar (bottom row, low ρ).

In [8]:
if HAVE_FULLRES:
    fig2 = save(icf.fig2(results), "fig2_spectra")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig2_spectra.pdf


### Figure 3 -- example velocity-vs-radius planar fits [B]

2x3 grid of v vs r_perp planar fits for the same six clouds as fig2, each panel annotated with its own ρ.

In [9]:
if HAVE_FULLRES:
    fig3 = save(icf.fig3(results), "fig3_profiles")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig3_profiles.pdf


### Figure 4 -- planarity summary [C]

Distribution of the Pearson planarity statistic ρ across all kept clouds, plus θ and j plotted against ρ.

In [10]:
if tab is not None:
    fig4 = save(icf.fig4(tab), "fig4_planarity")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig4_planarity.pdf


### Figure 5 -- Ω, j, M and gradient-PA distributions [C]

HI (red) vs CO GMC (blue) histograms of Ω, j, M and gradient position angle, this work (filled) overplotted with Mekuria & Imara (2026) Table 2 (unfilled step).

In [11]:
if tab is not None:
    fig5 = save(icf.fig5(tab, gmc, mi), "fig5_quantity_distributions")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig5_quantity_distributions.pdf


### Figure 6 -- per-cloud HI/GMC ratios [B]

Per-cloud log ratios of Ω, j and M (HI envelope over CO GMC), plus the folded gradient-axis offset ΔΦ.

In [12]:
if tab is not None:
    fig6 = save(icf.fig6(tab, gmc), "fig6_hi_gmc_ratios")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig6_hi_gmc_ratios.pdf


### Figure 7 -- Ω vs cloud radius [D]

Ω vs cloud radius, HI at R_HI and CO at R_GMC (this work, filled), with Imara/Bigiel/Blitz (2011) Table 1 overplotted as open markers, and size histograms along the top.

In [13]:
if tab is not None:
    fig7 = save(icf.fig7(tab, gmc, ibb), "fig7_omega_vs_size")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig7_omega_vs_size.pdf


### Figure 8 -- j vs cloud size [C]

log-log j vs R with power-law fits: CO GMCs (left, R_GMC) and HI envelopes (right, R_HI; ρ <= 0.75 systems drawn open and excluded from the fit).

In [14]:
if tab is not None:
    fig8 = save(icf.fig8(tab, gmc), "fig8_j_vs_size")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig8_j_vs_size.pdf


### Figure 9 -- specific angular momentum histograms

Histograms of j for the HI envelopes vs the CO GMCs.

In [15]:
if tab is not None:
    fig9 = save(icf.fig9(tab, gmc), "fig9_j_histograms")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig9_j_histograms.pdf


### Figure 10 -- rotational energy fraction histograms

Histograms of β_rot = Ω²R³/(3GM) (uniform-sphere, solid-body) for the HI envelopes and the CO GMCs.

In [16]:
if tab is not None:
    fig10 = save(icf.fig10(tab, gmc), "fig10_beta_rot_histograms")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig10_beta_rot_histograms.pdf


### Figure 11 -- R_HI vs the IBB 2011 HI-gradient extent

This pipeline's R_HI (fixed at 70 pc, the box half-width) vs IBB 2011's own fitted HI-gradient extent, one point per cloud, with the 1:1 line.

In [17]:
if tab is not None:
    fig11 = save(icf.fig11(tab, ibb), "fig11_rhi_vs_ibb_extent")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig11_rhi_vs_ibb_extent.pdf


### Figure 12 -- direct comparison against IBB 2011, Table 1

Per-cloud scatter of this work's Ω_HI, θ_HI and M_HI against IBB 2011's Table 1, for both the full-res and mid-res box-cutout runs, with grey connectors between each cloud's two points and open markers for ρ_HI < 0.75.

In [18]:
if tab is not None:
    fig12 = save(icf.fig12(tab, tab_mid, ibb), "fig12_vs_ibb_direct")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig12_vs_ibb_direct.pdf


### Figure 13 -- per-cloud profile view

1x3 stacked panels (Ω_HI, θ_HI, M_HI) vs GMC number, ordered by descending mid-res/full-res M_HI ratio: IBB 2011 (circle), this work full-res (triangle) and mid-res (square) at each cloud.

In [19]:
if tab is not None:
    fig13 = save(icf.fig13(tab, tab_mid, ibb), "fig13_per_cloud_profiles")
else:
    print("skipped -- needs the full-res cube")


wrote ../paper/figures_ibb2011_comparisons/fig13_per_cloud_profiles.pdf
